## Adaptation of the PLIMSTEX code by the Gross lab to analyse HDX data of GPCRs +/- $G_i \alpha$
By Matthew Chee in the context of Lab 1 internship in the Granier-Mouillac team at IGF

From previous experiments it is known that the binding stoichiometry of both receptors to $G_i \alpha$ is 1:1. The model will therefore be fitted according to this stoichiometry without considering the cases of 1:n as was done in the original paper. The code is in principle reusable for other protein-protein binding affinities as long as the binding is 1:1.

In [95]:
# libraries - run this cell first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.integrate import solve_ivp
from sklearn.metrics import r2_score
import os

### Parameters to be set by the user

Change these 3 sections for different data sets. There is no need to modify the rest of the code, unless you wish to further optimise it. 

#### 1. Load the data in CSV format

Load the Cluster file exported from DynamX. You can also renumber the residues if necessary and choose if you want to normalise the deuteration by the maximum deuteration (from the maxD data). 

In [96]:
data = pd.read_csv('cluster_KD_GiaACKR3.csv') # replace with the cluster file path: pd.read_csv('path/to/your/data')

renumber = -32 # set the offset in AA numbering (ex. -10 if you want the first residue in your sequence to start at -9)

maxD_exists = True # set to False if there is no maxD data

normalise_maxD = False # Set to True if you want to normalise the data by maxD

#### 2. Set parameters for fitting

Set the total protein concentration (in µM) in the deuterated mixture (the protein is the one kept constant, while the ligand is the one whose concentration varies). For example, if I mix 3 µL of my protein at 20 µM with 57 µL of labelling buffer, $Ptot = 20 \times \frac{3}{60} = 1 µM$.

The model for 1:1 binding is as follows:

$$ D(K, D_0, \Delta D_1, [L_T]) = D_0 - \Delta D_1 \cfrac{[PL]}{[P_T]} $$

$D$, the amount of deuteration (defined as the mass shift between $t_1$ and $t_0$), depends on the maximum deuteration when no ligand is present ($D_0$), the binding constant $K$ (which affects the $[PL]$), as well as $\Delta D_1$ which is the maximum difference in amount of deuteration (i.e. difference between the state with no ligand and the state saturated with ligand).

The initial estimates are important for proper fitting. The $K_D$ should be in µM, $D_0$ and $\Delta D_1$ should be in Da (if you are not normalising with the MaxD) or in % uptake (if normalising with the MaxD). If specific peptides show poor fitting when analysing the results, change the initial estimates and run the fitting for those peptides again.

The outlier threshold represents an arbitrary "tolerance" for outliers. The larger it is, the more tolerant you are of outliers and replicates that are further away from their "siblings" will be retained. Empirically, 2 seems to be a good value. The code does not remove any "outliers" if less than 3 replicates are present. It also does not remove outliers from the t0 or MaxD data, as the user is expected to have excluded peptides with poor t0 and MaxD spectra during data analysis in DynamX. 

MC is the number of pseudo-bootstrap iterations that will be performed to obtain an average and a standard deviation for the $K_D$ value. Each iteration involves selecting a random replicate for each concentration of ligand test and performing the fitting. 

In [97]:
do_ODE = False # set to 

Ptot = 0.775 # total protein concentration in µM in the deuterated sample

KD_init = 0.1 # best initial estimate of the dissociation constant in µM
D0_init = 1 # best initial estimate of the deuteration amount when no ligand is present (0 equivalents)
dD1_init = 1 # best initial estimate of the difference in deuteration between D0 and the fully saturated system

outlier_threshold = 2 # how tolerant you are of outliers in the data, the higher the number, the more tolerant you are

MC = 100 # number of iterations for the pseudo-bootstrapping of each peptide

#### 3. Choose where you want the fitted curves to be downloaded

If the folder does not already exist, it will be created

In [98]:
output = 'output' # path to the output folder, where results will be saved

#### Import HDX data

In [99]:
df = data
os.makedirs(output, exist_ok=True) # create folder to store PLIMSTEX fit images

# renumber residues if necessary
df['Start'] = df['Start'] + renumber
df['End'] = df['End'] + renumber 

# create a new series counting the number of occurrences of each charge state for each peptide
# in case the user forgets to deselect the other charge states in the DynamX
most_frequent_z = df.groupby('Sequence')['z'].apply(lambda x: x.value_counts().idxmax())

# create a boolean mask: for each row, check if the 'z' value in that row is equal to the most frequent 'z' for that peptide
df = df[df.apply(lambda row: row['z'] == most_frequent_z[row['Sequence']], axis=1)]

df['Exposure'] = df['Exposure'].astype(str)

if maxD_exists:
    df.loc[df['Exposure'] == list(df['Exposure'].unique())[0], 'Exposure'] = 't0'
    df.loc[df['Exposure'] == list(df['Exposure'].unique())[1], 'Exposure'] = 'maxD'
    df.loc[df['Exposure'] == list(df['Exposure'].unique())[2], 'Exposure'] = '0'

else:
    df.loc[df['Exposure'] == list(df['Exposure'].unique())[0], 'Exposure'] = 't0'
    df.loc[df['Exposure'] == list(df['Exposure'].unique())[1], 'Exposure'] = '0'

C:\Users\uid-2134\AppData\Local\Temp\ipykernel_11752\613636545.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Exposure'] = df['Exposure'].astype(str)


In [100]:
# calculate average t0 for each sequence
t0_all = []
for i in df['Sequence'].unique():
  t0_cum = []
  for j in df[(df['Sequence'] == i) & (df['Exposure'] == 't0')]['Center']:
    t0_cum.append(j)
    t0_avg = np.mean(t0_cum)
  t0_all.append(t0_avg)

# create a list of unique peptides and ligands
peptide_column = list(df['Sequence'].unique())
ligand_eq = list(df['Exposure'].unique())[2:]

peptide_data = {}

# create framework for peptide data
for i in peptide_column:
    ligand_dict = {j:[] for j in ligand_eq}
    peptide_data[i] = ligand_dict

# fill in the peptide data
for i in peptide_column:
    current_seq = df[df['Sequence'] == i]
    for j in ligand_eq:
        current_eq = current_seq[current_seq['Exposure'] == j]
        for k in current_eq['Center']:
            peptide_data[i][j].append(k - t0_all[peptide_column.index(i)])

In [101]:
if normalise_maxD:
    # loop to calculate average maxD for each sequence
    maxD_all = []
    for i in df['Sequence'].unique():
        maxD_cum = []
        for j in df[(df['Sequence'] == i) & (df['Exposure'] == 'maxD')]['Center']:
            maxD_cum.append(j)
            maxD_avg = np.mean(maxD_cum)
        maxD_all.append(maxD_avg)

    # normalise the data by maxD
    peptide_data_maxD = {}

    # create framework for normalised peptide data
    for i in peptide_column:
        ligand_dict = {j:[] for j in ligand_eq}
        peptide_data_maxD[i] = ligand_dict

    # fill in the normalised peptide data
    for i in peptide_column:
        current_seq = df[df['Sequence'] == i]
        for j in ligand_eq:
            current_eq = current_seq[current_seq['Exposure'] == j]
            for k in current_eq['Center']:
                norm_value = 100 * (k - t0_all[peptide_column.index(i)]) / (maxD_all[peptide_column.index(i)] - t0_all[peptide_column.index(i)])
                peptide_data_maxD[i][j].append(norm_value)
    
    peptide_data = peptide_data_maxD

#### Model creation
The model for 1:1 binding is as follows:

$$ D(K, D_0, \Delta D_1, [L_T]) = D_0 - \Delta D_1 \cfrac{[PL]}{[P_T]} $$

In order to calculate $\cfrac{[PL]}{[P_T]}$, we need to calculate $[PL]$ at the steady-state of deuteration (since $[P_T]$ is known - we choose the concentration of protein we add). We can find $[PL]$ if we know the $[L_{free}]$ (since $[PL] = [L_T] - [L_{free}]$ for 1:1 binding). $[L_{free}]$ depends on the binding  constant $K$, $[P_T]$, $[L_T]$. The original authors decide to numerically solve the following ODE in order to find $[L_{free}]$ from the value of $[L_T]$ (which is the known independent value we decide).

$$ \cfrac{d[L_{free}]}{d[L_T]} = \cfrac{1 + K[L_{free}]}{1 + 2K[L_{free}] + K[P_T] - K[L_T]} $$

This may not be necessary in a 1:1 binding situation and I have implemented an alternative below that allows for analytically solving for $[L_{free}]$.

Knowing that $K_D = \cfrac{[P_{free}][L_{free}]}{[PL]}$, $[PL] = [L_T] - [L_{free}]$ and $[P_{free}] = [P_T] - [PL]$, we have:

$$ K_D = \cfrac{([P_T] - [PL])[L_{free}]}{[PL]} $$
$$ K_D = \cfrac{([P_T] - ([L_T] - [L_{free}]))[L_{free}]}{([L_T] - [L_{free}])} $$
$$ K_D \cdot [L_T] - K_D \cdot [L_{free}] = ([P_T] - [L_T]) \cdot [L_{free}] + [L_{free}]^2 $$
$$ [L_{free}]^2 + ([P_T] - [L_T] + K_D) \cdot [L_{free}] - K_D \cdot [L_T] = 0 $$

We can ignore the negative root since the concentration has to be positive. Hence:

$$ [L_{free}] = \cfrac{-([P_T] - [L_T] + K_D) + \sqrt{([P_T] - [L_T] + K_D)^2 + 4 \cdot [K_D] \cdot [L_T]}}{2} $$

In [102]:
# define model to calculate deuteration as a function of total ligand concentration

if do_ODE:
  def freeligand(Ltot, Lfree, Ptot, KD):
    dLfreedLtot = (KD + Lfree) / (KD + 2*Lfree + Ptot - Ltot)
    return dLfreedLtot

  def totaldeut(Ltot, Ptot, D0, deltaD1, KD):
    # solve ODE to determine Lfree for each value of Ltot
    Lfree_init = 0 # if Ltot is 0, then obviously Lfree is 0
    Ltot_span = [Ltot[0], Ltot[-1]]
    Ltot_eval = np.linspace(Ltot_span[0], Ltot_span[1], 1000)

    sol = solve_ivp(freeligand, t_span=Ltot_span, y0=[Lfree_init],
                    t_eval=Ltot_eval, args=(Ptot, KD), method='Radau', dense_output=True)

    # create lookup lists with Lfree values for each Ltot value
    Lfree_lookup = sol.y[0]
    Ltot_lookup = sol.t
    Deut = []
    # check which Ltot value in lookup list is closest to actual Ltot and return corresponding Lfree
    Lfree_est = []
    for L in Ltot:
      idx = (np.abs(Ltot_lookup - L)).argmin()
      Lfree_est.append(Lfree_lookup[idx])
    Lfree_est = np.array(Lfree_est)
    Deut = D0 - deltaD1 * ((Ltot - Lfree_est) / Ptot)
    return Deut

else:
  def totaldeut(Ltot, Ptot, D0, deltaD1, KD):
    Ptot = float(Ptot)
    D0 = float(D0)
    deltaD1 = float(deltaD1)
    KD = float(KD)
    Lfree = []
    for L in Ltot:
      L = float(L)
      Lfree_calc = ((L-KD-Ptot)+np.sqrt((Ptot-L+KD)**2 + 4*KD*L))/2
      Lfree.append(float(Lfree_calc))
    Lfree = np.array(Lfree)
    Ltot = np.array(Ltot)
    Deut = D0 - deltaD1 * ((Ltot - Lfree) / Ptot)
    return Deut

#### Fitting of model to experimental data
We will fit the model to the experimental HDX data using a non-linear least squares (NLLS) regression. Residuals are the differences between experimentally measured deuteration and the expected deuteration according to the model at each $[L_T]$.

In [103]:
# params should be (D0, deltaD1, KD)
# model is defined in totaldeut()
def mse(params, model, data, Ltot_exp, Ptot):
  predict = model(Ltot_exp, Ptot, params[0], params[1], params[2])
  sq_err = [(predict[i] - data[i])**2 for i in range(len(predict))]
  return np.mean(sq_err)


# params is a list of the initial estimates
# myargs is a tuple of the model, data, Ltot_exp and Ptot
def param_estimate(params, myargs):
  init_est = np.array(params)
  res = minimize(mse, init_est, args=myargs, method='L-BFGS-B', bounds=[(None,None),(None,None),(0,None)])

  return res.x, res.fun
# res.x should contain 3 values - D0, deltaD1 and KD

In [104]:
# define a function to remove outliers
def remove_outliers(peptide, threshold):
  threshold = float(threshold)
  clean_data = []

  for i in peptide_data[peptide].values():
    i = tuple(sorted(i)) # sorts in ascending order
    if len(i) != 3: # don't remove replicates if there are only 2
      clean_data.append(i)
    else:
      rep1 = i[0]
      rep2 = i[1]
      rep3 = i[2]
      dist_a = abs(rep1-rep2)
      dist_b = abs(rep2-rep3)
      dist_c = abs(rep1-rep3)

      if dist_a == 0:
          dist_a += 1e-8
      if dist_b == 0:
          dist_b += 1e-8
      if dist_c == 0:
          dist_c += 1e-8

      if ((dist_c/dist_a) - (dist_c/dist_b)) > threshold:
        clean_data.append((rep1, rep2))
      elif ((dist_c/dist_a) - (dist_c/dist_b)) < -threshold:
        clean_data.append((rep2, rep3))
      else:
        clean_data.append(i)

  return clean_data

In [ ]:
peptide_num = 0

for peptide, data in peptide_data.items():
# Iterate through the peptide data
  ligand_conc_eqs = [float(eq) for eq in list(data.keys())]
  ligand_conc_eqs = np.array(ligand_conc_eqs)
  ligand_conc_abs = [float(eq) * Ptot for eq in list(data.keys())] # convert to absolute concentrations in µM
  ligand_conc_abs = np.array(ligand_conc_abs)
  init_estimates = [D0_init, dD1_init, KD_init] # initial estimates for D0, deltaD1 and KD

  cleaned_data = remove_outliers(data, outlier_threshold)

  KD_list = []
  r2_list = []
  for i in range(MC):
      deuteration_exp = np.array([np.random.choice(t) for t in cleaned_data])

      estimate = param_estimate(init_estimates, (totaldeut, deuteration_exp, ligand_conc_abs, Ptot))
      
      # plot data and fitted curve for visualisation
      D0_est = estimate[0][0]
      deltaD1_est = estimate[0][1]
      KD_est = estimate[0][2]
      KD_list.append(KD_est)
      
      deut_est_plotted = totaldeut(np.linspace(ligand_conc_abs[0],ligand_conc_abs[-1]), Ptot, D0_est, deltaD1_est, KD_est)
      deut_est = totaldeut(ligand_conc_abs, Ptot, D0_est, deltaD1_est, KD_est)
      
      r_squared = r2_score(deuteration_exp, deut_est)
      r2_list.append(r_squared)

      plt.plot(ligand_conc_eqs, deuteration_exp, 'o', label='data', color='black', alpha=0.5)
      plt.plot(np.linspace(ligand_conc_eqs[0],ligand_conc_eqs[-1]), deut_est_plotted, '-', label='fit', alpha=0.75)
      plt.xlabel('[Ligand total]/[Protein total]')
      if normalise_maxD:
        plt.ylabel('Relative deuteration (% of max D)')
      else:
        plt.ylabel('Absolute deuteration (Da)')
      plt.title(f'Fitted PLIMSTEX curve for {peptide}')

  KD_avg = np.average(KD_list)
  KD_sd = np.std(KD_list)
  r2_avg = np.average(r2_list)

  plt.text(0.5, -0.15, f"Average R-squared: {r2_avg:.3f}\nAverage KD: {KD_avg:.3f} ± {KD_sd:.3f} µM",
          horizontalalignment='center',
          verticalalignment='top',
          transform=plt.gca().transAxes) # Use transAxes to position text relative to the axes

  # Save the plot to a PNG file
  pep_start = str(list(set(df[df['Sequence']==peptide]['Start'].unique()))[0])
  pep_end = str(list(set(df[df['Sequence']==peptide]['End'].unique()))[0])
  peptide_name = pep_start+"-"+pep_end+"_"+peptide
  plt.savefig(f'{output}/PLIMSTEX fit for {peptide_name}.png', bbox_inches='tight') # Use bbox_inches='tight' to include the text
  peptide_num += 1
  print(peptide_num)
  # Optionally, clear the current figure
  plt.clf()

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61


<Figure size 640x480 with 0 Axes>